# Project SHOES Models V2

Small language model, on-device shoe classification model.

This notebook does:
- load in data
- feature selection
- model training
- optimizer model for generating predictions (needed?)
- hyperparameter tuning

Application needs to be able to:
- predict the condition
- predict the materials
- predict / recommend the "next chapter"
- classify the brand
  - percentage match
  - user can input yes or no for "reinforcement"
- build up the database and model(s) via users


## Resources
- https://www.kaggle.com/code/prthmgoyl/efficientnet-resnet-vgg-shoes/input

In [ ]:
!pip install -U pillow
!pip install -U transformers
!pip install -U trl
!pip install -U datasets
!pip install -U accelerate
!pip install -U huggingface_hub
!pip install -q faiss-cpu open-clip-torch

  Using cached transformers-5.9.0-py3-none-any.whl.metadata (33 kB)
Using cached transformers-5.9.0-py3-none-any.whl (10.8 MB)
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 760.8/760.8 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 14.3 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.2/668.2 kB 16.4 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existi

In [ ]:
import torch
import json
import random
import os
import pandas as pd
import shutil
import tensorflow as tf
from sklearn.model_selection import train_test_split

from pprint import pprint
from datasets import load_dataset

from PIL import Image
from transformers import AutoProcessor # Keeping for potential use with HuggingFace vision models like SigLIP/FashionCLIP

from tqdm.notebook import tqdm
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import kagglehub # Keeping as it might be used for data access

# Device setup
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[INFO] Using device: {device}")
if device == "cuda":
    print(f"[INFO] GPU: {torch.cuda.get_device_name(0)}")
    print(f"[INFO] GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

[INFO] Using device: cuda
[INFO] GPU: Tesla T4
[INFO] GPU Memory: 15.6 GB


In [ ]:
from google.colab import drive


# Check if the mount point exists and is not empty, then unmount if possible
# or clear the directory if it's not actually a mounted drive.
if os.path.exists('/content/drive') and os.path.ismount('/content/drive'):
    print('Drive is already mounted.')
elif os.path.exists('/content/drive') and len(os.listdir('/content/drive')) > 0:
    print('Mount point /content/drive contains files but is not mounted. Attempting to clear directory...')
    # !!! WARNING: This will DELETE all contents in /content/drive if it's not a mounted drive.
    # Use with caution.
    try:
        for item in os.listdir('/content/drive'):
            item_path = os.path.join('/content/drive', item)
            if os.path.isfile(item_path) or os.path.islink(item_path):
                os.unlink(item_path)
            elif os.path.isdir(item_path):
                shutil.rmtree(item_path)
        print("Directory /content/drive cleared successfully.")
    except Exception as e:
        print(f"Error clearing /content/drive: {e}")
        print("To resolve this, you might still need to restart your Colab runtime.")

# Attempt to mount the drive
drive.mount('/content/drive')

Drive is already mounted.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Logging

setup logging.

In [ ]:
result_path = 'result_storage'
base_path = '/content/drive/MyDrive/SHOES/'

# Create result_path if it doesn't exist
if not os.path.exists(result_path):
    os.makedirs(result_path)
    print(f"Created directory: {result_path}")
else:
    print(f"Directory already exists: {result_path}")

# Check if base_path exists
if os.path.exists(base_path):
    print(f"Base path exists: {base_path}")
else:
    print(f"WARNING: Base path does not exist: {base_path}. Please ensure your Google Drive is correctly mounted and the path is valid.")

Created directory: result_storage
Base path exists: /content/drive/MyDrive/SHOES/


# Load data

There is currently a data source for each classification type, but perhaps these can be combined in the future in an intelligent way.

In [ ]:
# hardcode full paths
material_class_indices_path = '/content/drive/MyDrive/SHOES/data/material_classification/material_class_indices.json'
material_data_path = '/content/drive/MyDrive/SHOES/data/material_classification/material_data.csv'
processed_material_images_path = '/content/drive/MyDrive/SHOES/processed_material_images'
material_train_path = '/content/drive/MyDrive/SHOES/data/material_classification/material_train.csv'
material_test_path = '/content/drive/MyDrive/SHOES/data/material_classification/material_test.csv'
material_val_path = '/content/drive/MyDrive/SHOES/data/material_classification/material_val.csv'

brand_class_indices_path = '/content/drive/MyDrive/SHOES/data/brand_classification/brand_class_indices.json'
brand_test_path = '/content/drive/MyDrive/SHOES/data/brand_classification/brand_test.csv'
brand_train_path = '/content/drive/MyDrive/SHOES/data/brand_classification/brand_train.csv'
brand_val_path = '/content/drive/MyDrive/SHOES/data/brand_classification/brand_val.csv'


print("\n--- Loading Material Data ---")
# Load material class indices
with open(material_class_indices_path, 'r') as f:
    material_class_indices = json.load(f)
print(f"Material Class Indices loaded: {material_class_indices}")

# Load material data CSVs
material_df = pd.read_csv(material_data_path)
print(f"Material Data (material_df) loaded. Shape: {material_df.shape}")
material_train_df = pd.read_csv(material_train_path)
print(f"Material Train Data (material_train_df) loaded. Shape: {material_train_df.shape}")
material_test_df = pd.read_csv(material_test_path)
print(f"Material Test Data (material_test_df) loaded. Shape: {material_test_df.shape}")
material_val_df = pd.read_csv(material_val_path)
print(f"Material Validation Data (material_val_df) loaded. Shape: {material_val_df.shape}")

print("\n--- Loading Brand Data ---")
# Load brand class indices
with open(brand_class_indices_path, 'r') as f:
    brand_class_indices = json.load(f)
print(f"Brand Class Indices loaded: {brand_class_indices}")

# Load brand data CSVs
brand_train_df = pd.read_csv(brand_train_path)
print(f"Brand Train Data (brand_train_df) loaded. Shape: {brand_train_df.shape}")
brand_test_df = pd.read_csv(brand_test_path)
print(f"Brand Test Data (brand_test_df) loaded. Shape: {brand_test_df.shape}")
brand_val_df = pd.read_csv(brand_val_path)
print(f"Brand Validation Data (brand_val_df) loaded. Shape: {brand_val_df.shape}")

print("\n--- Sample DataFrames ---")
print("Material Data Head:")
pprint(material_df.head())
print("\nBrand Train Data Head:")
pprint(brand_train_df.head())


--- Loading Material Data ---
Material Class Indices loaded: {'Cable Knit': 0, 'Canvas': 1, 'Canvas;Cordura;Cotton;Lace;Leather;Rubber': 2, 'Canvas;Cotton': 3, 'Canvas;Cotton;Lace;Rubber': 4, 'Canvas;Cotton;Leather': 5, 'Canvas;Cotton;Leather;Rubber;Suede': 6, 'Canvas;Cotton;Leather;Rubber;Suede;Synthetic': 7, 'Canvas;Cotton;Rubber': 8, 'Canvas;Cotton;Synthetic': 9, 'Canvas;Faux Leather;Mesh;Suede;Synthetic': 10, 'Canvas;Full-grain leather': 11, 'Canvas;Lace': 12, 'Canvas;Lace;Leather;Rubber': 13, 'Canvas;Lace;Rubber': 14, 'Canvas;Lace;Synthetic': 15, 'Canvas;Leather': 16, 'Canvas;Leather;Rubber': 17, 'Canvas;Leather;Rubber;Suede': 18, 'Canvas;Leather;Suede': 19, 'Canvas;Leather;Tweed': 20, 'Canvas;Rubber': 21, 'Canvas;Rubber;Suede': 22, 'Canvas;Rubber;Synthetic': 23, 'Canvas;Shearling': 24, 'Canvas;Suede': 25, 'Canvas;Synthetic': 26, 'Cordura;Leather;Nubuck': 27, 'Cordura;Nylon;Rubber': 28, 'Corduroy': 29, 'Cork': 30, 'Cork;Leather': 31, 'Cotton': 32, 'Cotton;Leather': 33, 'Deerskin;

# Split Data

In [ ]:
def split_data(df, train_ratio, validation_ratio, test_ratio):
    x_train, x_test = train_test_split(df, test_size=1 - train_ratio, stratify=df['Material'])
    x_val, x_test = train_test_split(x_test, test_size=test_ratio/(test_ratio + validation_ratio), stratify=x_test['Material'])

    print(f'Shape of Training Data : ', x_train.shape)
    print(f'Shape of Testing Data : ', x_test.shape)
    print(f'Shape of Validation Data : ', x_val.shape)

    # Ensure x_test is a copy to avoid SettingWithCopyWarning later
    x_test = x_test.copy()

    return x_train, x_val, x_test

# Example usage of the new function
train_ratio = 0.7
validation_ratio = 0.15
test_ratio = 0.15

x_train_df, x_val_df, x_test_df = split_data(material_df, train_ratio, validation_ratio, test_ratio)

datagen = ImageDataGenerator(
    rescale=1/255.,
    zoom_range=0.2,
    rotation_range=30,
    horizontal_flip=True
)


def create_image_generators(train_df, test_df, val_df, img_size):
    img_datagen = ImageDataGenerator(preprocessing_function=tf.keras.applications.mobilenet_v2.preprocess_input)

    train_generator = img_datagen.flow_from_dataframe(
        dataframe=train_df,
        x_col='ImagePaths',
        y_col='Material',
        target_size=img_size,
        color_mode='rgb',
        class_mode='categorical',
        batch_size=32,
        seed=42
    )
    test_generator = img_datagen.flow_from_dataframe(
        dataframe=test_df,
        x_col='ImagePaths',
        y_col='Material',
        target_size=img_size,
        color_mode='rgb',
        class_mode='categorical',
        batch_size=32,
        seed=42
    )
    val_generator = img_datagen.flow_from_dataframe(
        dataframe=val_df,
        x_col='ImagePaths',
        y_col='Material',
        target_size=img_size,
        color_mode='rgb',
        class_mode='categorical',
        batch_size=32,
        seed=42
    )
    return train_generator, test_generator, val_generator

#Pre-Processing #4
img_size=(224, 224)

train_generator, test_generator, val_generator = create_image_generators(x_train_df, x_test_df, x_val_df, img_size)

Shape of Training Data :  (31586, 2)
Shape of Testing Data :  (6769, 2)
Shape of Validation Data :  (6769, 2)
Found 31586 validated image filenames belonging to 232 classes.
Found 6769 validated image filenames belonging to 232 classes.
Found 6769 validated image filenames belonging to 232 classes.


# Load Models

In [ ]:
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
import numpy as np
import uuid # Import uuid for generating unique event IDs
import tensorflow as tf # Ensure tf is imported for mobilenet_v2.preprocess_input

def usage(img_path):
    # Generate a unique event ID for this inference
    event_id = str(uuid.uuid4())

    # Load models
    brand_model = load_model('/content/drive/MyDrive/SHOES/models/brand_vgg16_model.h5')
    material_model = load_model('/content/drive/MyDrive/SHOES/models/material_custom_model.h5')

    # Load and preprocess image for both models
    img = image.load_img(img_path, target_size=(224, 224))
    img_array = image.img_to_array(img)

    # Preprocess for brand model
    img_array_brand = np.expand_dims(img_array, axis=0)
    img_array_brand = img_array_brand / 255.0  # Rescale for brand model

    # Preprocess for material model
    img_array_material = np.expand_dims(img_array, axis=0)
    img_array_material = tf.keras.applications.mobilenet_v2.preprocess_input(img_array_material)

    # Get class names for brand from the globally loaded dictionary
    brand_class_names = [k for k, v in sorted(brand_class_indices.items(), key=lambda item: item[1])]

    # Get class names for material from the globally loaded dictionary
    material_class_names = [k for k, v in sorted(material_class_indices.items(), key=lambda item: item[1])]

    # Predict brand
    brand_preds = brand_model.predict(img_array_brand)
    brand_confidence = np.max(brand_preds) * 100
    predicted_brand_idx = np.argmax(brand_preds)
    predicted_brand = brand_class_names[predicted_brand_idx]

    # Predict material
    material_preds = material_model.predict(img_array_material)
    material_confidence = np.max(material_preds) * 100
    predicted_material_idx = np.argmax(material_preds)
    predicted_material = material_class_names[predicted_material_idx]

    # Refactor to match Target Inference Output Schema
    return {
        "event_id": event_id,
        "image": img_path,
        "brand": {
            "label": predicted_brand,
            "confidence": float(round(brand_confidence / 100, 2)), # Convert to 0-1 scale and explicitly cast to float
            "evidence": [] # Placeholder for future retrieval evidence
        },
        "model": {
            "label": None, # Placeholder for future model detection
            "confidence": None,
            "neighbors_used": None
        },
        "material": {
            "label": predicted_material,
            "confidence": float(round(material_confidence / 100, 2)) # Convert to 0-1 scale and explicitly cast to float
        },
        "action": {
            "label": None, # Placeholder for future decision layer
            "confidence": None,
            "reason": None
        }
    }

In [ ]:
# Get an example image path from the training dataframe
example_img_path = x_train_df['ImagePaths'].iloc[0]

# Call the usage function with the example image path
prediction_results = usage(example_img_path)

# Print the results
print(prediction_results)

1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 832ms/step
{'event_id': '940c7007-8eda-475f-a32c-a24bb4afd39a', 'image': '/content/drive/MyDrive/SHOES/processed_material_images/7679203.369194.jpg', 'brand': {'label': 'nike', 'confidence': 0.9900000095367432, 'evidence': []}, 'model': {'label': None, 'confidence': None, 'neighbors_used': None}, 'material': {'label': 'Leather', 'confidence': 0.23999999463558197}, 'action': {'label': None, 'confidence': None, 'reason': None}}


# Target System

## Layer 1: Embedding Model

* Use pretrained vision-language model (SigLIP / CLIP / FashionCLIP)
* Convert each image into a dense vector embedding

In [ ]:
# ============================================================
# STEP 1: GENERATE EMBEDDINGS FOR SHOE IMAGES
# ============================================================

# !pip install -q transformers torch pillow

import torch
from PIL import Image
from transformers import AutoProcessor, AutoModel
import numpy as np
import pandas as pd
from tqdm import tqdm

# ------------------------------------------------------------
# LOAD PRETRAINED EMBEDDING MODEL
# SigLIP is a strong modern choice for image similarity
# ------------------------------------------------------------

MODEL_NAME = "google/siglip-base-patch16-224"

processor = AutoProcessor.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

# ------------------------------------------------------------
# FUNCTION: GENERATE EMBEDDING FROM IMAGE
# ------------------------------------------------------------

def generate_embedding(img_path):

    image = Image.open(img_path).convert("RGB")

    inputs = processor(
        images=image,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = model.get_image_features(**inputs)

    # The traceback indicates outputs is a BaseModelOutputWithPooling object.
    # Access the pooled output, which is the desired single embedding vector.
    embedding = outputs.pooler_output.cpu().numpy()

    # At this point, embedding should be (1, hidden_dim). The following checks
    # and squeeze ensure it's a 1D vector (hidden_dim,).
    if embedding.ndim == 3: # Should not be hit if pooler_output is accessed
        embedding = embedding.mean(axis=1)
    elif embedding.ndim == 2 and embedding.shape[0] > 1: # Should not be hit if pooler_output is accessed
        embedding = embedding.mean(axis=0)

    # Ensure it's a 1D vector (remove batch_size dimension if present, e.g., (1, 768) -> (768,))
    embedding = embedding.squeeze()

    # normalize embedding
    embedding = embedding / np.linalg.norm(embedding)

    return embedding

# ------------------------------------------------------------
# TEST ON SINGLE IMAGE
# ------------------------------------------------------------


test_embedding = generate_embedding(example_img_path)

print("Embedding shape:", test_embedding.shape)

# ------------------------------------------------------------
# GENERATE EMBEDDINGS FOR ENTIRE DATASET
# Assumes dataframe has column: ImagePaths
# ------------------------------------------------------------

embeddings = []

# --- MODIFIED: Sample the material_df for quicker testing ---
sample_size = 1000 # You can adjust this number
sampled_material_df = material_df.sample(n=sample_size, random_state=42) # Using random_state for reproducibility

for img_path in tqdm(sampled_material_df["ImagePaths"], desc="Generating embeddings for sampled data"):

    try:
        emb = generate_embedding(img_path)

        embeddings.append(emb)

    except Exception as e:

        print(f"Error processing {img_path}: {e}")

        embeddings.append(None)

# ------------------------------------------------------------
# STORE EMBEDDINGS IN DATAFRAME
# ------------------------------------------------------------

sampled_material_df["embedding"] = embeddings

print("Done.")
print(sampled_material_df[["ImagePaths", "embedding"]].head())

Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

Embedding shape: (768,)


Generating embeddings for sampled data: 100%|██████████| 1000/1000 [00:17<00:00, 56.11it/s]

Done.
                                              ImagePaths  \
39089  /content/drive/MyDrive/SHOES/processed_materia...   
11960  /content/drive/MyDrive/SHOES/processed_materia...   
18633  /content/drive/MyDrive/SHOES/processed_materia...   
5262   /content/drive/MyDrive/SHOES/processed_materia...   
31895  /content/drive/MyDrive/SHOES/processed_materia...   

                                               embedding  
39089  [0.015357012, 0.026600115, 0.010462653, -0.014...  
11960  [0.025484435, 0.004773682, -0.0062125875, -0.0...  
18633  [-0.0027121622, -0.0090551935, -0.028063677, -...  
5262   [-0.008027241, 0.011110158, -0.0001809186, -0....  
31895  [0.036515042, 0.020496633, 0.030763678, 0.0198...  


In [ ]:
# save embeddings

import os

# Define the target directory for saving (base_path + data)
save_dir = os.path.join(base_path, 'data')

# Create the directory if it doesn't exist
os.makedirs(save_dir, exist_ok=True)
print(f"Ensured directory exists: {save_dir}")

# Save the sampled_material_df with embeddings to a CSV file
embeddings_output_path = os.path.join(save_dir, 'sampled_material_embeddings.csv')
sampled_material_df.to_csv(embeddings_output_path, index=False)
print(f"Sampled material embeddings saved to: {embeddings_output_path}")

Ensured directory exists: /content/drive/MyDrive/SHOES/data
Sampled material embeddings saved to: /content/drive/MyDrive/SHOES/data/sampled_material_embeddings.csv


In [ ]:
# ============================================================
# STEP 2: BUILD FAISS RETRIEVAL INDEX
# ============================================================

!pip install -q faiss-cpu

import faiss
import numpy as np

# ------------------------------------------------------------
# REMOVE FAILED EMBEDDINGS
# ------------------------------------------------------------

# Use sampled_material_df instead of material_df as it contains the embeddings
material_df_with_embeddings = sampled_material_df[sampled_material_df["embedding"].notnull()].reset_index(drop=True)

# ------------------------------------------------------------
# STACK EMBEDDINGS INTO NUMPY MATRIX
# Shape:
# (num_images, embedding_dim)
# ------------------------------------------------------------

embedding_matrix = np.vstack(material_df_with_embeddings["embedding"].values).astype("float32")

print("Embedding matrix shape:", embedding_matrix.shape)

# ------------------------------------------------------------
# CREATE FAISS INDEX
# Using cosine similarity via inner product
# ------------------------------------------------------------

embedding_dim = embedding_matrix.shape[1]

index = faiss.IndexFlatIP(embedding_dim)

# embeddings already normalized earlier
index.add(embedding_matrix)

print("Total embeddings indexed:", index.ntotal)

# ------------------------------------------------------------
# FUNCTION: FIND SIMILAR SHOES
# ------------------------------------------------------------

def find_similar_shoes(query_embedding, top_k=5):

    query_embedding = np.array([query_embedding]).astype("float32")

    similarities, indices = index.search(query_embedding, top_k)

    results = []

    for sim, idx in zip(similarities[0], indices[0]):

        # Use material_df_with_embeddings for retrieval
        row = material_df_with_embeddings.iloc[idx]

        results.append({
            "similarity": float(sim),
            "image_path": row["ImagePaths"],
            "brand": row.get("brand", "unknown"), # 'brand' column is not available in sampled_material_df
            "material": row["Material"] # Correctly access the 'Material' column
        })

    return results

# ------------------------------------------------------------
# TEST RETRIEVAL
# ------------------------------------------------------------

# Use material_df_with_embeddings for testing retrieval
test_embedding = material_df_with_embeddings.iloc[0]["embedding"]

results = find_similar_shoes(test_embedding, top_k=5)

for r in results:
    print(r)

Embedding matrix shape: (1000, 768)
Total embeddings indexed: 1000
{'similarity': 1.0000001192092896, 'image_path': '/content/drive/MyDrive/SHOES/processed_material_images/8013962.82.jpg', 'brand': 'unknown', 'material': 'Leather;Nappa'}
{'similarity': 0.8517781496047974, 'image_path': '/content/drive/MyDrive/SHOES/processed_material_images/8015840.4418.jpg', 'brand': 'unknown', 'material': 'Leather'}
{'similarity': 0.8409013748168945, 'image_path': '/content/drive/MyDrive/SHOES/processed_material_images/8021425.103681.jpg', 'brand': 'unknown', 'material': 'Synthetic'}
{'similarity': 0.8397107124328613, 'image_path': '/content/drive/MyDrive/SHOES/processed_material_images/7762904.89.jpg', 'brand': 'unknown', 'material': 'Patent Leather'}
{'similarity': 0.8300806879997253, 'image_path': '/content/drive/MyDrive/SHOES/processed_material_images/7859424.3.jpg', 'brand': 'unknown', 'material': 'Leather'}


In [ ]:
# ============================================================
# STEP 3: RETRIEVAL-BASED INFERENCE
# ============================================================

from collections import Counter

# ------------------------------------------------------------
# AGGREGATE LABELS FROM NEIGHBORS
# ------------------------------------------------------------

def aggregate_prediction(results, field="brand"):

    labels = []
    weights = []

    for r in results:

        label = r[field]

        similarity = r["similarity"]

        labels.append(label)

        weights.append(similarity)

    # weighted voting
    score_dict = {}

    for label, weight in zip(labels, weights):

        if label not in score_dict:
            score_dict[label] = 0

        score_dict[label] += weight

    # best label
    best_label = max(score_dict, key=score_dict.get)

    # confidence
    total_score = sum(score_dict.values())

    confidence = score_dict[best_label] / total_score

    return {
        "label": best_label,
        "confidence": round(float(confidence), 3),
        "all_scores": score_dict
    }

# ------------------------------------------------------------
# FULL RETRIEVAL INFERENCE PIPELINE
# ------------------------------------------------------------

def retrieval_inference(img_path, top_k=10):

    # generate embedding
    query_embedding = generate_embedding(img_path)

    # retrieve neighbors
    neighbors = find_similar_shoes(query_embedding, top_k=top_k)

    # aggregate predictions
    brand_prediction = aggregate_prediction(neighbors, field="brand")

    material_prediction = aggregate_prediction(neighbors, field="material")

    return {
        "image": img_path,
        "brand": brand_prediction,
        "material": material_prediction,
        "neighbors": neighbors
    }

# ------------------------------------------------------------
# TEST
# ------------------------------------------------------------

# Get an example image path from the sampled_material_df
example_retrieval_img_path = sampled_material_df['ImagePaths'].iloc[0]
result = retrieval_inference(example_retrieval_img_path)

print("\nBRAND PREDICTION")
print(result["brand"])

print("\nMATERIAL PREDICTION")
print(result["material"])

print("\nTOP NEIGHBORS")
for n in result["neighbors"]:
    print(n)


BRAND PREDICTION
{'label': 'unknown', 'confidence': 1.0, 'all_scores': {'unknown': 8.390032410621643}}

MATERIAL PREDICTION
{'label': 'Leather', 'confidence': 0.297, 'all_scores': {'Leather;Nappa': 1.0000001192092896, 'Leather': 2.4906874895095825, 'Synthetic': 1.6375467777252197, 'Patent Leather': 1.6671556234359741, 'Suede': 0.7975181341171265, 'Cotton': 0.7971242666244507}}

TOP NEIGHBORS
{'similarity': 1.0000001192092896, 'image_path': '/content/drive/MyDrive/SHOES/processed_material_images/8013962.82.jpg', 'brand': 'unknown', 'material': 'Leather;Nappa'}
{'similarity': 0.8517781496047974, 'image_path': '/content/drive/MyDrive/SHOES/processed_material_images/8015840.4418.jpg', 'brand': 'unknown', 'material': 'Leather'}
{'similarity': 0.8409013748168945, 'image_path': '/content/drive/MyDrive/SHOES/processed_material_images/8021425.103681.jpg', 'brand': 'unknown', 'material': 'Synthetic'}
{'similarity': 0.8397107124328613, 'image_path': '/content/drive/MyDrive/SHOES/processed_materi

In [ ]:
# ============================================================
# STEP 4: COMPARE CNN VS RETRIEVAL
# ============================================================

comparison_results = []

# ------------------------------------------------------------
# LOOP THROUGH SAMPLE IMAGES
# ------------------------------------------------------------

sample_size = 25

# Sample from sampled_material_df to ensure images have embeddings in the FAISS index
sample_df = sampled_material_df.sample(sample_size, random_state=42)

for _, row in sample_df.iterrows():

    img_path = row["ImagePaths"]

    try:

        # ----------------------------------------------------
        # CNN PREDICTION
        # your existing function
        # ----------------------------------------------------

        cnn_result = usage(img_path)

        # ----------------------------------------------------
        # RETRIEVAL PREDICTION
        # ----------------------------------------------------

        retrieval_result = retrieval_inference(img_path)

        # ----------------------------------------------------
        # STORE COMPARISON
        # ----------------------------------------------------

        comparison_results.append({

            "image": img_path,

            # CNN
            "cnn_brand": cnn_result["brand"]["label"],
            "cnn_brand_conf": cnn_result["brand"]["confidence"],

            "cnn_material": cnn_result["material"]["label"],
            "cnn_material_conf": cnn_result["material"]["confidence"],

            # Retrieval
            "retrieval_brand": retrieval_result["brand"]["label"],
            "retrieval_brand_conf": retrieval_result["brand"]["confidence"],

            "retrieval_material": retrieval_result["material"]["label"],
            "retrieval_material_conf": retrieval_result["material"]["confidence"]

        })

    except Exception as e:

        print(f"Error processing {img_path}: {e}")

# ------------------------------------------------------------
# RESULTS DATAFRAME
# ------------------------------------------------------------

comparison_df = pd.DataFrame(comparison_results)

print(comparison_df.head())

# ------------------------------------------------------------
# AGREEMENT ANALYSIS
# ------------------------------------------------------------

comparison_df["brand_agreement"] = (
    comparison_df["cnn_brand"] ==
    comparison_df["retrieval_brand"]
)

comparison_df["material_agreement"] = (
    comparison_df["cnn_material"] ==
    comparison_df["retrieval_material"]
)

print("\nBRAND AGREEMENT RATE:")
print(comparison_df["brand_agreement"].mean())

print("\nMATERIAL AGREEMENT RATE:")
print(comparison_df["material_agreement"].mean())

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 939ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 407ms/step


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 953ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 437ms/step


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 609ms/step


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 946ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 393ms/step


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 967ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 405ms/step


...

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 636ms/step


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 416ms/step


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 952ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 411ms/step


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 444ms/step


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 430ms/step
                                               image cnn_brand  \
0  /content/drive/MyDrive/SHOES/processed_materia...    adidas   
1  /content/drive/MyDrive/SHOES/processed_materia...      nike   
2  /content/drive/MyDrive/SHOES/processed_materia...    adidas   
3  /content/drive/MyDrive/SHOES/processed_materia...      nike   
4  /content/drive/MyDrive/SHOES/processed_materia...      nike   

   cnn_brand_conf    cnn_material  cnn_material_conf retrieval_brand  \
0            0.87         Leather               0.83         unknown   
1            0.94         Leather               0.82         unknown   
2            0.88         Leather               0.20         unknown   
3            0.88       Synthetic               0.50         unknown   
4            0.99  Mesh;Synthetic               0.35         unknown   

   retrieval_brand_conf retrieval_material  retrieval_material_conf  
0                   1.0  

In [ ]:
import pandas as pd
import numpy as np
import faiss
import torch
from PIL import Image
from transformers import AutoProcessor, AutoModel
import os
import json
import uuid
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image

# --- PATHS AND GLOBAL VARIABLES ---
base_path = '/content/drive/MyDrive/SHOES/'
embeddings_output_path = os.path.join(base_path, 'data', 'sampled_material_embeddings.csv')
material_class_indices_path = os.path.join(base_path, 'data', 'material_classification', 'material_class_indices.json')
brand_class_indices_path = os.path.join(base_path, 'data', 'brand_classification', 'brand_class_indices.json')
brand_model_path = os.path.join(base_path, 'models', 'brand_vgg16_model.h5')
material_model_path = os.path.join(base_path, 'models', 'material_custom_model.h5')

# Device setup
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load class indices (needed for CNN usage function)
with open(material_class_indices_path, 'r') as f:
    material_class_indices = json.load(f)
material_class_names = [k for k, v in sorted(material_class_indices.items(), key=lambda item: item[1])]

with open(brand_class_indices_path, 'r') as f:
    brand_class_indices = json.load(f)
brand_class_names = [k for k, v in sorted(brand_class_indices.items(), key=lambda item: item[1])]

# --- EMBEDDING MODEL SETUP (SigLIP) ---
MODEL_NAME = "google/siglip-base-patch16-224"
processor = AutoProcessor.from_pretrained(MODEL_NAME)
embedding_model = AutoModel.from_pretrained(MODEL_NAME).to(device)

def generate_embedding(img_path):
    image_pil = Image.open(img_path).convert("RGB")
    inputs = processor(images=image_pil, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = embedding_model.get_image_features(**inputs)
    embedding = outputs.pooler_output.cpu().numpy().squeeze()
    embedding = embedding / np.linalg.norm(embedding)
    return embedding

# --- FAISS INDEX SETUP ---
# Load the pre-computed embeddings
sampled_material_df_loaded = pd.read_csv(embeddings_output_path)

# Function to parse the embedding strings back into NumPy arrays
def parse_embedding_string(embedding_str):
    # Remove brackets and split by space, then convert to float
    return np.array([float(val) for val in embedding_str.replace('[', '').replace(']', '').split() if val], dtype=np.float32)

# Apply the parsing function
sampled_material_df_loaded['embedding'] = sampled_material_df_loaded['embedding'].apply(parse_embedding_string)

material_df_with_embeddings = sampled_material_df_loaded[sampled_material_df_loaded["embedding"].notnull()].reset_index(drop=True)
embedding_matrix = np.vstack(material_df_with_embeddings["embedding"].values).astype("float32")

embedding_dim = embedding_matrix.shape[1]
faiss_index = faiss.IndexFlatIP(embedding_dim)
faiss_index.add(embedding_matrix)

def find_similar_shoes(query_embedding, top_k=5):
    query_embedding = np.array([query_embedding]).astype("float32")
    similarities, indices = faiss_index.search(query_embedding, top_k)
    results = []
    for sim, idx in zip(similarities[0], indices[0]):
        row = material_df_with_embeddings.iloc[idx]
        results.append({
            "similarity": float(sim),
            "image_path": row["ImagePaths"],
            "brand": row.get("brand", "unknown"),
            "material": row["Material"]
        })
    return results

def aggregate_prediction(results, field="brand"):
    labels = []
    weights = []
    for r in results:
        label = r[field]
        similarity = r["similarity"]
        labels.append(label)
        weights.append(similarity)
    score_dict = {}
    for label, weight in zip(labels, weights):
        if label not in score_dict:
            score_dict[label] = 0
        score_dict[label] += weight
    best_label = max(score_dict, key=score_dict.get)
    total_score = sum(score_dict.values())
    confidence = score_dict[best_label] / total_score
    return {
        "label": best_label,
        "confidence": round(float(confidence), 3),
        "all_scores": score_dict
    }

def retrieval_inference(img_path, top_k=10):
    query_embedding = generate_embedding(img_path)
    neighbors = find_similar_shoes(query_embedding, top_k=top_k)
    brand_prediction = aggregate_prediction(neighbors, field="brand")
    material_prediction = aggregate_prediction(neighbors, field="material")
    return {
        "image": img_path,
        "brand": brand_prediction,
        "material": material_prediction,
        "neighbors": neighbors
    }

# --- CNN MODEL SETUP AND INFERENCE FUNCTION ---
# Load models globally for efficiency
brand_model = load_model(brand_model_path)
material_model = load_model(material_model_path)

def cnn_inference(img_path):
    event_id = str(uuid.uuid4())
    img = image.load_img(img_path, target_size=(224, 224))
    img_array = image.img_to_array(img)

    img_array_brand = np.expand_dims(img_array, axis=0)
    img_array_brand = img_array_brand / 255.0

    img_array_material = np.expand_dims(img_array, axis=0)
    img_array_material = tf.keras.applications.mobilenet_v2.preprocess_input(img_array_material)

    brand_preds = brand_model.predict(img_array_brand, verbose=0)
    brand_confidence = np.max(brand_preds)
    predicted_brand_idx = np.argmax(brand_preds)
    predicted_brand = brand_class_names[predicted_brand_idx]

    material_preds = material_model.predict(img_array_material, verbose=0)
    material_confidence = np.max(material_preds)
    predicted_material_idx = np.argmax(material_preds)
    predicted_material = material_class_names[predicted_material_idx]

    return {
        "event_id": event_id,
        "image": img_path,
        "brand": {
            "label": predicted_brand,
            "confidence": float(round(brand_confidence, 2)),
            "evidence": []
        },
        "model": {
            "label": None,
            "confidence": None,
            "neighbors_used": None
        },
        "material": {
            "label": predicted_material,
            "confidence": float(round(material_confidence, 2))
        },
        "action": {
            "label": None,
            "confidence": None,
            "reason": None
        }
    }

# --- COMBINED INFERENCE FUNCTION ---
def combined_inference(img_path, top_k_retrieval=10):
    cnn_results = cnn_inference(img_path)
    retrieval_results = retrieval_inference(img_path, top_k=top_k_retrieval)

    combined_output = {
        "image": img_path,
        "cnn_predictions": {
            "brand": cnn_results["brand"],
            "material": cnn_results["material"]
        },
        "retrieval_predictions": {
            "brand": retrieval_results["brand"],
            "material": retrieval_results["material"]
        },
        "retrieved_neighbors": retrieval_results["neighbors"]
    }
    return combined_output

print("All inference functions and models loaded and ready for use.")

Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

All inference functions and models loaded and ready for use.


In [ ]:
# Load and Merge Data for Brand Prediction

# Reuse sampled_material_df_loaded which already has parsed embeddings
# and brand_train_df loaded from previous steps.

# Merge the dataframes to enrich sampled_material_df_loaded with brand information
# A left merge ensures all entries from sampled_material_df_loaded are kept.
material_df_with_embeddings_and_brands = pd.merge(
    sampled_material_df_loaded,
    brand_train_df[['ImagePaths', 'Labels']],
    on='ImagePaths',
    how='left'
)

# Rename the 'Labels' column to 'Brand' for clarity and consistency
material_df_with_embeddings_and_brands = material_df_with_embeddings_and_brands.rename(columns={'Labels': 'Brand'})

# Handle missing brand information by filling NaN values with 'unknown'
material_df_with_embeddings_and_brands['Brand'] = material_df_with_embeddings_and_brands['Brand'].fillna('unknown')

# --- TEMPORARY FIX: Assign some 'known' brands for demonstration ---
# The merge above likely resulted in all 'unknown' because image paths don't overlap.
# To allow testing, we'll manually assign some brands to a few random rows.
# In a real scenario, you would ensure your material and brand datasets have overlapping image paths or are integrated.

# Identify rows that are currently 'unknown' to assign new brands to a subset
unknown_brand_indices = material_df_with_embeddings_and_brands[material_df_with_embeddings_and_brands['Brand'] == 'unknown'].index

if not unknown_brand_indices.empty:
    # Ensure there are enough unknown brands to assign to. If not, reduce the number of assignments.
    num_to_assign_nike = min(5, len(unknown_brand_indices))
    nike_indices = np.random.choice(unknown_brand_indices, num_to_assign_nike, replace=False)
    material_df_with_embeddings_and_brands.loc[nike_indices, 'Brand'] = 'nike'

    # Recalculate remaining unknown indices for adidas assignment
    remaining_unknown_indices = material_df_with_embeddings_and_brands[material_df_with_embeddings_and_brands['Brand'] == 'unknown'].index
    num_to_assign_adidas = min(5, len(remaining_unknown_indices))
    adidas_indices = np.random.choice(remaining_unknown_indices, num_to_assign_adidas, replace=False)
    material_df_with_embeddings_and_brands.loc[adidas_indices, 'Brand'] = 'adidas'

print("Merged DataFrame with embeddings and brands head:")
print(material_df_with_embeddings_and_brands.head())
print("\nNumber of entries with known brands (nike/adidas):", material_df_with_embeddings_and_brands[material_df_with_embeddings_and_brands['Brand'] != 'unknown'].shape[0])
print("\nMerged DataFrame Info:")
material_df_with_embeddings_and_brands.info()

Merged DataFrame with embeddings and brands head:
                                          ImagePaths                Material  \
0  /content/drive/MyDrive/SHOES/processed_materia...           Leather;Nappa   
1  /content/drive/MyDrive/SHOES/processed_materia...  Faux Leather;Synthetic   
2  /content/drive/MyDrive/SHOES/processed_materia...               Synthetic   
3  /content/drive/MyDrive/SHOES/processed_materia...                 Leather   
4  /content/drive/MyDrive/SHOES/processed_materia...               Synthetic   

                                           embedding    Brand  
0  [0.015357012, 0.026600115, 0.010462653, -0.014...  unknown  
1  [0.025484435, 0.004773682, -0.0062125875, -0.0...  unknown  
2  [-0.0027121622, -0.0090551935, -0.028063677, -...  unknown  
3  [-0.008027241, 0.011110158, -0.0001809186, -0....  unknown  
4  [0.036515042, 0.020496633, 0.030763678, 0.0198...  unknown  

Number of entries with known brands (nike/adidas): 10

Merged DataFrame Info:
<class

## Rebuild FAISS Index with Brand Information

Now that we have merged the brand information with the material embeddings, we need to rebuild the FAISS index to include these new labels. This will allow the `retrieval_inference` function to return brand predictions, instead of just 'unknown'.

In [ ]:
# Update the global DataFrame used by the FAISS retrieval functions
material_df_with_embeddings = material_df_with_embeddings_and_brands.copy()

# Re-extract embeddings from the updated DataFrame
embedding_matrix = np.vstack(material_df_with_embeddings["embedding"].values).astype("float32")

print("Rebuilt embedding matrix shape:", embedding_matrix.shape)

# Re-initialize FAISS index
faiss_index = faiss.IndexFlatIP(embedding_dim)

# Add the new embeddings to the index
faiss_index.add(embedding_matrix)

print("Total embeddings indexed in the new FAISS index:", faiss_index.ntotal)

print("FAISS index successfully rebuilt with brand information.")

Rebuilt embedding matrix shape: (1000, 768)
Total embeddings indexed in the new FAISS index: 1000
FAISS index successfully rebuilt with brand information.


## Test Enhanced Retrieval Brand Prediction

Now, let's test the `combined_inference` function to see if the retrieval system is now predicting brands. We will use an example image path and inspect the output, paying close attention to the `retrieval_predictions.brand` field.

In [ ]:
# Get an example image path from the DataFrame that has embeddings and brands
# We'll pick one that ideally has a known brand for demonstration
# Now that we've ensured some brands are present, this should no longer error.
example_image_with_brand_path = material_df_with_embeddings[material_df_with_embeddings['Brand'] != 'unknown']['ImagePaths'].iloc[0]

if example_image_with_brand_path:
    print(f"Testing combined inference with image: {example_image_with_brand_path}")
    combined_inference_results = combined_inference(example_image_with_brand_path, top_k_retrieval=10)

    print("\n--- Combined Inference Results ---")
    pprint(combined_inference_results)

    print("\nRetrieval Brand Prediction:")
    print(combined_inference_results['retrieval_predictions']['brand'])
    print("\nRetrieval Material Prediction:")
    print(combined_inference_results['retrieval_predictions']['material'])
else:
    # This else block should ideally not be hit with the temporary fix in place
    print("Could not find an example image with a known brand in the sampled dataset.")
    print("Please ensure 'material_df_with_embeddings' contains images with 'Brand' information other than 'unknown'.")

Testing combined inference with image: /content/drive/MyDrive/SHOES/processed_material_images/7222237.326572.jpg

--- Combined Inference Results ---
{'cnn_predictions': {'brand': {'confidence': 0.5600000023841858,
                               'evidence': [],
                               'label': 'adidas'},
                     'material': {'confidence': 0.8999999761581421,
                                  'label': 'Nylon;Polyester;Rubber'}},
 'image': '/content/drive/MyDrive/SHOES/processed_material_images/7222237.326572.jpg',
 'retrieval_predictions': {'brand': {'all_scores': {'unknown': 7.976569294929504},
                                     'confidence': 1.0,
                                     'label': 'unknown'},
                           'material': {'all_scores': {'Canvas': 0.7498329877853394,
                                                       'Cork': 0.7447309494018555,
                                                       'Leather': 1.490128219127655,
            

# Improve Data

# Task
The user wants to improve the `Project SHOES Models V2` notebook which focuses on building an on-device shoe classification model. The goal is to enhance the model's ability to predict shoe condition, materials, recommend 'next chapter', and classify brands, including reinforcement learning from user input. The current notebook loads data, performs feature selection, model training, and hyperparameter tuning. The immediate next steps involve analyzing brand data discrepancies, evaluating combined CNN and retrieval inference performance, proposing a robust brand data integration strategy, and outlining training data improvement areas.

## Analyze Brand Data Discrepancy

### Subtask:
Examine the image paths in `sampled_material_df_loaded` and `brand_train_df` to understand why brand information is predominantly 'unknown' in the merged `material_df_with_embeddings_and_brands`. This will involve identifying if there's an overlap in image paths or a fundamental mismatch preventing proper brand integration.


**Reasoning**:
First, I'll inspect the image paths in both dataframes to understand their structure and content. Then, I'll calculate the number of unique image paths in each DataFrame and determine the overlap to understand the data discrepancy.



In [ ]:
print("--- ImagePaths in sampled_material_df_loaded ---")
print(sampled_material_df_loaded['ImagePaths'].head())
print(f"Total unique image paths in sampled_material_df_loaded: {sampled_material_df_loaded['ImagePaths'].nunique()}")

print("\n--- ImagePaths in brand_train_df ---")
print(brand_train_df['ImagePaths'].head())
print(f"Total unique image paths in brand_train_df: {brand_train_df['ImagePaths'].nunique()}")

# Identify the overlap
material_image_paths = set(sampled_material_df_loaded['ImagePaths'])
brand_image_paths = set(brand_train_df['ImagePaths'])

overlap_paths = material_image_paths.intersection(brand_image_paths)

print(f"\nNumber of overlapping image paths between sampled_material_df_loaded and brand_train_df: {len(overlap_paths)}")

--- ImagePaths in sampled_material_df_loaded ---
0    /content/drive/MyDrive/SHOES/processed_materia...
1    /content/drive/MyDrive/SHOES/processed_materia...
2    /content/drive/MyDrive/SHOES/processed_materia...
3    /content/drive/MyDrive/SHOES/processed_materia...
4    /content/drive/MyDrive/SHOES/processed_materia...
Name: ImagePaths, dtype: object
Total unique image paths in sampled_material_df_loaded: 1000

--- ImagePaths in brand_train_df ---
0    /kaggle/input/nike-adidas-shoes-for-image-clas...
1    /kaggle/input/nike-adidas-shoes-for-image-clas...
2    /kaggle/input/nike-adidas-shoes-for-image-clas...
3    /kaggle/input/nike-adidas-shoes-for-image-clas...
4    /kaggle/input/nike-adidas-shoes-for-image-clas...
Name: ImagePaths, dtype: object
Total unique image paths in brand_train_df: 460

Number of overlapping image paths between sampled_material_df_loaded and brand_train_df: 0
